# Adapter Evaluation

Generate lyrics from trained adapters and classify with RoBERTa.

In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer as AT
from peft import PeftModel

model_path = "./models/gemma-4-E4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

clf_path = "./classifier_output/best_model"
clf_model = AutoModelForSequenceClassification.from_pretrained(clf_path)
clf_tokenizer = AT.from_pretrained(clf_path)
clf_model.eval()
labels = clf_model.config.id2label

In [ ]:
def generate_lyrics(model, prompt="Write song lyrics.\n\n", max_new_tokens=512, min_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_new_tokens=min_new_tokens,
        temperature=0.9,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )
    return tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)


def classify(text):
    enc = clf_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        logits = clf_model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
    return {labels[j]: probs[j].item() for j in range(len(labels))}


def evaluate_adapter(adapter_path, n_samples=10):
    model = PeftModel.from_pretrained(base_model, adapter_path)
    results = []
    samples = []
    for i in range(n_samples):
        text = generate_lyrics(model)
        samples.append(text)
        probs = classify(text)
        results.append(probs)
        top = max(probs, key=probs.get)
        print(f"  Sample {i+1}: {top} ({probs[top]:.3f})")
    model.unload()
    return samples, pd.DataFrame(results)

## Adapter results

In [ ]:
adapters = {
    "Gojira (LoRA r=8)": "./adapters/gojira",
    "Tool (DoRA r=8)": "./adapters/tool",
}

all_results = {}

for name, path in adapters.items():
    print(f"\n=== {name} ===")
    samples, df = evaluate_adapter(path)
    all_results[name] = {"samples": samples, "df": df}
    print(f"\nMean attribution:\n{df.mean().round(4)}")
    print(f"Std:\n{df.std().round(4)}")

## Summary

In [ ]:
rows = []
for name, data in all_results.items():
    mean = data["df"].mean()
    std = data["df"].std()
    row = {"Adapter": name}
    for artist in ["Death", "Gojira", "Meshuggah", "Opeth", "Tool"]:
        row[artist] = f"{mean[artist]:.3f} +/- {std[artist]:.3f}"
    rows.append(row)

summary = pd.DataFrame(rows).set_index("Adapter")
print(summary.to_string())